# NG.L Backtest — Multi-Strategy Evaluation with vectorbt

Strategies tested on `data/ta-data/NG.L.csv`:
1. **SuperTrend** — Entry on direction flip (-1→1), exit on (1→-1)
2. **RSI Reversal** — Buy RSI<30, sell RSI>70
3. **MACD Histogram** — Buy when histogram turns positive, sell when negative
4. **MA Crossover** — Buy MA20>MA50, sell MA20<MA50
5. **Structure** — Buy HH+HL (bullish structure), sell LL+LH (bearish)

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import vectorbt as vbt
from vectorbt.portfolio.enums import SizeType
from IPython.display import display

# Configure Plotly renderer to avoid nbformat mime-type check
import plotly.io as pio
pio.renderers.default = "jupyterlab"


In [2]:
# === 1. Imports & Setup ===
import pandas as pd
from pathlib import Path

# --- Setup ---
DATA_DIR = Path("../data/ta-data")

def load_ticker(ticker: str) -> pd.DataFrame:
    """Load TA-enriched CSV for a given ticker."""
    csv_path = DATA_DIR / f"{ticker}.csv"
    df = pd.read_csv(csv_path, parse_dates=["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    return df

# Default: NG.L
ticker = "NG.L"
df = load_ticker(ticker)
df.index = df["Date"]

# --- Filter to last 10 years ---
cutoff_date = pd.Timestamp("2024-10-15")
df = df[df["Date"] >= cutoff_date].reset_index(drop=True)
df.index = df["Date"]

print(f"Ticker: {ticker}, Rows: {len(df)}, Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

Ticker: NG.L, Rows: 507, Date range: 2024-10-15 to 2026-10-15


In [3]:
df.tail()

,Date,Open,High,Low,Close,Volume,ta_ema20,ta_ema50,ta_ema200,ta_supertrend,...,ta_lh,ta_ll,ta_trend,ta_bos,ta_choch,ta_nearest_resistance,ta_nearest_support,ta_return_5d,ta_return_10d,ta_return_20d
Date,,,,,,,,,,,,,,,,,,,,,
2026-10-09,2026-10-09,1168.5,1168.5,1147.0,1147.0,14349058,1144.839455,1161.322800,1191.756369,545.428588,...,0,0,1,0,0,1158.5,1100.500000,NaN,NaN,NaN
2026-10-12,2026-10-12,1156.0,1164.5,1149.5,1164.5,5564139,1146.711887,1161.447396,1191.485162,1310.512490,...,0,0,1,0,0,1158.5,1100.500000,NaN,NaN,NaN
2026-10-13,2026-10-13,1175.0,1187.5,1168.5,1178.5,9803585,1149.739327,1162.116126,1191.355956,545.428588,...,0,0,1,0,0,1215.5,1100.500000,NaN,NaN,NaN
2026-10-14,2026-10-14,1182.0,1195.0,1175.5,1185.5,7992900,1153.145105,1163.033140,1191.297688,1310.512490,...,0,0,1,0,0,1215.5,1100.500000,NaN,NaN,NaN
2026-10-15,2026-10-15,1200.0,1210.5,1186.5,1193.5,12896784,1156.988428,1164.227919,1191.319602,545.428588,...,0,0,1,0,0,1215.5,1185.908325,NaN,NaN,NaN


In [4]:
def signal_supertrend(df: pd.DataFrame):
    direction = df["ta_supertrend_direction"]
    entry = (direction == 1) & (direction.shift(1) == -1)
    exit_ = (direction == -1) & (direction.shift(1) == 1)
    return entry.astype(int), exit_.astype(int) * -1

entry_st, exit_st = signal_supertrend(df)
print(f"SuperTrend signals: {entry_st.sum()} entries, {abs(exit_st).sum()} exits")

# Visualize signals on price
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    row_heights=[0.7, 0.3], subplot_titles=["Price & SuperTrend", "Direction"])
fig.add_trace(go.Candlestick(x=df.index, open=df["Open"], high=df["High"],
                             low=df["Low"], close=df["Close"], name="Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_supertrend_upper"], line=dict(color="#ff7f0e", width=1), name="Upper"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_supertrend_lower"], line=dict(color="#2ca02c", width=1), name="Lower"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_supertrend_direction"], line=dict(color="#1f77b4", width=1)), row=2, col=1)
fig.update_layout(height=600, title="SuperTrend Signals", showlegend=True)
display(fig)


SuperTrend signals: 222 entries, 223 exits


In [5]:
def signal_supertrend(df: pd.DataFrame):
    direction = df["ta_supertrend_direction"]
    entry = (direction == 1) & (direction.shift(1) == -1)
    exit_ = (direction == -1) & (direction.shift(1) == 1)
    return entry.astype(int), exit_.astype(int) * -1

entry_st, exit_st = signal_supertrend(df)
print(f"SuperTrend signals: {entry_st.sum()} entries, {abs(exit_st).sum()} exits")

# Visualize signals on price
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    row_heights=[0.7, 0.3], subplot_titles=["Price & SuperTrend", "Direction"])
fig.add_trace(go.Candlestick(x=df.index, open=df["Open"], high=df["High"],
                             low=df["Low"], close=df["Close"], name="Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_supertrend_upper"], line=dict(color="#ff7f0e", width=1), name="Upper"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_supertrend_lower"], line=dict(color="#2ca02c", width=1), name="Lower"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_supertrend_direction"], line=dict(color="#1f77b4", width=1)), row=2, col=1)
fig.update_layout(height=600, title="SuperTrend Signals", showlegend=True)
display(fig)

SuperTrend signals: 222 entries, 223 exits


## 2. RSI Reversal Strategy

Buy when RSI < 30 (oversold), sell when RSI > 70 (overbought).

In [6]:
def signal_rsi(df: pd.DataFrame):
    rsi = df["ta_rsi14"]
    entry = rsi < 30
    exit_ = rsi > 70
    return entry.astype(int), exit_.astype(int) * -1


entry_rsi, exit_rsi = signal_rsi(df)
print(f"RSI signals: {entry_rsi.sum()} entries, {abs(exit_rsi).sum()} exits")

# Visualize RSI signals on price
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    row_heights=[0.7, 0.3], subplot_titles=["Price & RSI", "RSI (14)"])
fig.add_trace(go.Candlestick(x=df.index, open=df["Open"], high=df["High"],
                             low=df["Low"], close=df["Close"], name="Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_rsi14"], line=dict(color="#1f77b4", width=1), name="RSI(14)"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=pd.Series(70, index=df.index), line=dict(color="#ff7f0e", width=1, dash="dash")), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=pd.Series(30, index=df.index), line=dict(color="#ff7f0e", width=1, dash="dash")), row=2, col=1)
fig.update_layout(height=600, title="RSI Reversal Signals", showlegend=True)
display(fig)

RSI signals: 4 entries, 46 exits


## 3. MACD Histogram Strategy

Buy when histogram turns positive, sell when negative.

In [7]:
def signal_macd(df: pd.DataFrame):
    hist = df["ta_macd_histogram"]
    entry = (hist > 0) & (hist.shift(1) <= 0)
    exit_ = (hist < 0) & (hist.shift(1) >= 0)
    return entry.astype(int), exit_.astype(int) * -1


entry_macd, exit_macd = signal_macd(df)
print(f"MACD signals: {entry_macd.sum()} entries, {abs(exit_macd).sum()} exits")

# Visualize MACD signals
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=["Price & MA", "MACD Line", "MACD Histogram"])
fig.add_trace(go.Candlestick(x=df.index, open=df["Open"], high=df["High"],
                             low=df["Low"], close=df["Close"], name="Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_ma20"], line=dict(color="#ff7f0e", width=1), name="MA20"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_ma50"], line=dict(color="#17becf", width=1), name="MA50"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_macd_line"], line=dict(color="#1f77b4", width=1), name="MACD"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_macd_signal"], line=dict(color="#ff7f0e", width=1), name="Signal"), row=2, col=1)
colors_hist = df["ta_macd_histogram"].apply(lambda x: "#2ca02c" if x >= 0 else "#d62728")
fig.add_trace(go.Bar(x=df.index, y=df["ta_macd_histogram"], marker_color=colors_hist, name="Histogram"), row=3, col=1)
fig.update_layout(height=800, title="MACD Histogram Signals", showlegend=True)
display(fig)

MACD signals: 24 entries, 23 exits


## 4. MA Crossover Strategy

Buy when MA20 crosses above MA50, sell when MA20 crosses below MA50.

In [8]:
def signal_ma_crossover(df: pd.DataFrame):
    ma20 = df["ta_ma20"]
    ma50 = df["ta_ma50"]
    entry = (ma20 > ma50) & (ma20.shift(1) <= ma50.shift(1))
    exit_ = (ma20 < ma50) & (ma20.shift(1) >= ma50.shift(1))
    return entry.astype(int), exit_.astype(int) * -1


entry_ma, exit_ma = signal_ma_crossover(df)
print(f"MA Crossover signals: {entry_ma.sum()} entries, {abs(exit_ma).sum()} exits")

# Visualize MA crossover signals
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    row_heights=[0.8, 0.2],
                    subplot_titles=["Price & MA20/MA50 Crossover", "MA Spread (MA20 - MA50)"])
fig.add_trace(go.Candlestick(x=df.index, open=df["Open"], high=df["High"],
                             low=df["Low"], close=df["Close"], name="Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_ma20"], line=dict(color="#ff7f0e", width=1), name="MA20"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_ma50"], line=dict(color="#17becf", width=1), name="MA50"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["ta_ma20"] - df["ta_ma50"], line=dict(color="#1f77b4", width=1), name="Spread"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=pd.Series(0, index=df.index), line=dict(color="black", width=0.5)), row=2, col=1)
fig.update_layout(height=700, title="MA Crossover Signals", showlegend=True)
display(fig)

MA Crossover signals: 7 entries, 8 exits


## 5. Structure-Based Strategy

Buy on HH+HL (bullish structure), sell on LL+LH (bearish structure).

In [9]:
def signal_structure(df: pd.DataFrame):
    hh = df["ta_hh"].fillna(0).astype(int)
    hl = df["ta_hl"].fillna(0).astype(int)
    ll = df["ta_ll"].fillna(0).astype(int)
    lh = df["ta_lh"].fillna(0).astype(int)

    entry = (hh == 1) | (hl == 1)
    exit_ = (ll == 1) | (lh == 1)
    return entry.astype(int), exit_.astype(int) * -1


entry_struct, exit_struct = signal_structure(df)
print(f"Structure signals: {entry_struct.sum()} entries, {abs(exit_struct).sum()} exits")

# Visualize structure signals
df_struct = df.copy()
df_struct["signal"] = entry_struct.astype(int) - exit_struct.abs().astype(int)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    row_heights=[0.7, 0.3], subplot_titles=["Price & Structure Signals", "Structure Markers"])
fig.add_trace(go.Candlestick(x=df.index, open=df["Open"], high=df["High"],
                             low=df["Low"], close=df["Close"], name="Price"), row=1, col=1)
# Mark buy signals
buy_mask = entry_struct.astype(bool)
fig.add_trace(go.Scatter(x=df.loc[buy_mask].index, y=df.loc[buy_mask, "Low"] * 0.995,
                         mode="markers", marker=dict(symbol="triangle-up", size=8, color="green"),
                         name="Buy"), row=1, col=1)
# Mark sell signals
sell_mask = (exit_struct.abs().astype(bool))
fig.add_trace(go.Scatter(x=df.loc[sell_mask].index, y=df.loc[sell_mask, "High"] * 1.005,
                         mode="markers", marker=dict(symbol="triangle-down", size=8, color="red"),
                         name="Sell"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df_struct["signal"], line=dict(color="#1f77b4", width=1)), row=2, col=1)
fig.update_layout(height=600, title="Structure-Based Signals", showlegend=True)
display(fig)

Structure signals: 20 entries, 27 exits


## Backtest Runner

Run a single strategy backtest using vectorbt.

In [10]:
STRATEGIES = {
    "supertrend": signal_supertrend,
    "rsi": signal_rsi,
    "macd": signal_macd,
    "ma_crossover": signal_ma_crossover,
    "structure": signal_structure,
}


def run_backtest(
    df: pd.DataFrame,
    strategy_name: str,
    init_cash: float = 100_000,
    commission: float = 0.001,  # 0.1% per trade
    slippage: float = 0.0,
) -> vbt.Portfolio:
    """Run a single backtest and return a vectorbt Portfolio object."""
    signals = STRATEGIES[strategy_name](df)
    entries, exits = signals

    pf = vbt.Portfolio.from_signals(
        close=df["Close"],
        entries=entries,
        exits=exits,
        init_cash=init_cash,
        freq="1D",
        slippage=slippage,
        size=1.0,
        size_type=SizeType.Percent,
        fees=commission,
    )
    return pf


def print_summary(pf: vbt.Portfolio, name: str):
    """Print a formatted summary for one strategy."""
    stats_df = pf.stats()
    total_return = stats_df.get("Total Return [%]", 0)
    total_trades = stats_df.get("Total Trades", 0)
    win_rate = stats_df.get("Win Rate [%]", 0)
    profit_factor = stats_df.get("Profit Factor", 0)
    max_dd = stats_df.get("Max Drawdown [%]", 0)
    final_value = pf.final_value()

    print(f"\n{'='*60}")
    print(f"  {name.upper()}")
    print(f"{'='*60}")
    print(f"  Total Return:      {total_return:7.2f}%")
    print(f"  Total Trades:      {total_trades:6d}")
    print(f"  Win Rate:          {win_rate:6.1f}%")
    print(f"  Profit Factor:     {profit_factor:7.3f}")
    print(f"  Max Drawdown:      {max_dd:7.2f}%")
    print(f"  Final Net Value:   £{final_value:,.0f}")

### Run a single strategy

In [11]:
# Pick a strategy to test
strategy_name = "ma_crossover"

pf = run_backtest(df, strategy_name)
print_summary(pf, strategy_name)

# Show detailed stats
pf.stats()


  MA_CROSSOVER
  Total Return:         3.30%
  Total Trades:           7
  Win Rate:            42.9%
  Profit Factor:       1.253
  Max Drawdown:        13.50%
  Final Net Value:   £103,297


Start                         2024-10-15 00:00:00
End                           2026-10-15 00:00:00
Period                          507 days 00:00:00
Start Value                              100000.0
End Value                           103297.432194
Total Return [%]                         3.297432
Benchmark Return [%]                    17.396344
Max Gross Exposure [%]                      100.0
Total Fees Paid                        1370.13516
Max Drawdown [%]                        13.499999
Max Drawdown Duration           177 days 00:00:00
Total Trades                                    7
Total Closed Trades                             7
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]                            42.857143
Best Trade [%]                           9.846288
Worst Trade [%]                         -6.446613
Avg Winning Trade [%]                    5.734237
Avg Losing Trade [%]                    -3.263361


### Compare all strategies

In [12]:
def compare_all(df: pd.DataFrame, strategies: list[str] | None = None):
    """Run all strategies and print a comparison table."""
    if strategies is None:
        strategies = list(STRATEGIES.keys())

    results = []
    for name in strategies:
        print(f"Running strategy: {name} ...")
        pf = run_backtest(df, name)
        print_summary(pf, name)

        stats_df = pf.stats()
        results.append({
            "Strategy": name,
            "Total Return (%)": round(stats_df.get("Total Return [%]", 0), 2),
            "Total Trades": int(stats_df.get("Total Trades", 0)),
            "Win Rate (%)": round(stats_df.get("Win Rate [%]", 0), 1),
            "Profit Factor": round(stats_df.get("Profit Factor", 0), 3),
            "Max Drawdown (%)": round(stats_df.get("Max Drawdown [%]", 0), 2),
            "Final Net Value": round(pf.final_value(), 0),
        })

    table = pd.DataFrame(results).set_index("Strategy")
    print(f"\n{'='*80}")
    print("  COMPARISON SUMMARY")
    print(f"{'='*80}")
    display(table)
    return table


table = compare_all(df)

Running strategy: supertrend ...

  SUPERTREND
  Total Return:       -25.57%
  Total Trades:         222
  Win Rate:            41.9%
  Profit Factor:       0.754
  Max Drawdown:        29.32%
  Final Net Value:   £74,428
Running strategy: rsi ...

  RSI
  Total Return:        12.30%
  Total Trades:           2
  Win Rate:           100.0%
  Profit Factor:         inf
  Max Drawdown:        18.16%
  Final Net Value:   £112,298
Running strategy: macd ...

  MACD
  Total Return:       -13.78%
  Total Trades:          24
  Win Rate:            30.4%
  Profit Factor:       0.622
  Max Drawdown:        23.59%
  Final Net Value:   £86,215
Running strategy: ma_crossover ...

  MA_CROSSOVER
  Total Return:         3.30%
  Total Trades:           7
  Win Rate:            42.9%
  Profit Factor:       1.253
  Max Drawdown:        13.50%
  Final Net Value:   £103,297
Running strategy: structure ...

  STRUCTURE
  Total Return:        30.49%
  Total Trades:          11
  Win Rate:            60.0%


,Total Return (%),Total Trades,Win Rate (%),Profit Factor,Max Drawdown (%),Final Net Value
Strategy,,,,,,
supertrend,-25.57,222,41.9,0.754,29.32,74428.0
rsi,12.30,2,100.0,inf,18.16,112298.0
macd,-13.78,24,30.4,0.622,23.59,86215.0
ma_crossover,3.30,7,42.9,1.253,13.50,103297.0
structure,30.49,11,60.0,9.698,12.09,130492.0


## Equity Curves & Drawdown

Plot cumulative returns and drawdown for all strategies.

In [13]:
def plot_comparison(df: pd.DataFrame, strategies: list[str] | None = None):
    """Plot equity curves using plotly directly."""
    if strategies is None:
        strategies = list(STRATEGIES.keys())

    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.7, 0.3],
        subplot_titles=["Cumulative Returns", "Drawdown"],
    )

    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

    for i, name in enumerate(strategies):
        pf = run_backtest(df, name)
        cum_ret = pf.cumulative_returns()
        fig.add_trace(
            go.Scatter(x=cum_ret.index, y=cum_ret.values, name=name,
                       line=dict(width=1.5, color=colors[i % len(colors)])),
            row=1, col=1,
        )
        dd = pf.drawdown()
        fig.add_trace(
            go.Scatter(x=dd.index, y=dd.values, name=name + " DD",
                       line=dict(width=1, color=colors[i % len(colors)]),
                       opacity=0.6),
            row=2, col=1,
        )

    fig.update_layout(
        title="NG.L Strategy Comparison",
        height=700,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
        hovermode="x unified",
    )

    display(fig)


plot_comparison(df)

## Trade Log

View the actual trade records for a selected strategy.

In [14]:
def show_trades(pf: vbt.Portfolio, name: str):
    """Display individual trades as a DataFrame."""
    trades = pf.trades
    print(f"{name} — {len(trades)} trades\n")
    display(trades.to_frame().T if hasattr(trades, 'to_frame') else trades)


pf_rsi = run_backtest(df, "rsi")
show_trades(pf_rsi, "RSI Reversal")

pf_st = run_backtest(df, "supertrend")
show_trades(pf_st, "SuperTrend")

RSI Reversal — 2 trades



SuperTrend — 222 trades



## Parameter Tuning

Try different parameter values for a strategy.

In [15]:
from itertools import product

def signal_rsi_tuning(df: pd.DataFrame, buy_threshold: int = 30, sell_threshold: int = 70):
    """RSI with configurable thresholds."""
    rsi = df["ta_rsi14"]
    entry = rsi < buy_threshold
    exit_ = rsi > sell_threshold
    return entry.astype(int), exit_.astype(int) * -1

# Grid search over RSI thresholds
buy_thresholds = [20, 25, 30, 35]
sell_thresholds = [65, 70, 75]

results_tune = []
for bt, st in product(buy_thresholds, sell_thresholds):
    if bt >= st:
        continue
    # Patch STRATEGIES temporarily
    old_rsi = STRATEGIES["rsi"]
    STRATEGIES["rsi"] = lambda df, b=bt, s=st: signal_rsi_tuning(df, b, s)

    pf = run_backtest(df, "rsi")
    stats = pf.stats()
    results_tune.append({
        "Buy Thresh": bt,
        "Sell Thresh": st,
        "Total Return (%)": round(stats.get("Total Return [%]", 0), 2),
        "Total Trades": int(stats.get("Total Trades", 0)),
        "Win Rate (%)": round(stats.get("Win Rate [%]", 0), 1),
        "Profit Factor": round(stats.get("Profit Factor", 0), 3),
        "Max Drawdown (%)": round(stats.get("Max Drawdown [%]", 0), 2),
    })
    STRATEGIES["rsi"] = old_rsi  # restore

tune_table = pd.DataFrame(results_tune)
display(tune_table.sort_values("Profit Factor", ascending=False))

,Buy Thresh,Sell Thresh,Total Return (%),Total Trades,Win Rate (%),Profit Factor,Max Drawdown (%)
7,30,70,12.30,2,100.0,inf,18.16
8,30,75,17.93,2,100.0,inf,18.16
11,35,75,9.37,2,100.0,inf,18.16
10,35,70,4.14,2,100.0,inf,18.16
6,30,65,12.19,2,50.0,7.608,18.16
9,35,65,4.04,2,50.0,2.112,18.16
0,20,65,0.00,0,NaN,NaN,NaN
1,20,70,0.00,0,NaN,NaN,NaN
2,20,75,0.00,0,NaN,NaN,NaN
3,25,65,0.00,0,NaN,NaN,NaN


## Save Results

In [16]:
import matplotlib
matplotlib.use('nbAgg')

# Patch matplotlib's font lookup: map Arial → Liberation Sans
import matplotlib.font_manager as fm_mod
from matplotlib.font_manager import FontProperties

_orig_findfont = fm_mod.findfont

def _patched_findfont(prop, *args, **kwargs):
    if isinstance(prop, str):
        prop = FontProperties(family=prop)
    if prop.get_family() and any('Arial' in f for f in prop.get_family()):
        prop.set_family([f if 'Arial' not in f else 'Liberation Sans' for f in prop.get_family()])
    return _orig_findfont(prop, *args, **kwargs)

fm_mod.findfont = _patched_findfont
fm_mod.fontManager.findfont = _patched_findfont

# Suppress IPython.display to avoid JS errors when quantstats tries to render inline
import sys
class _Suppress:
    def write(self, s): pass
    def flush(self): pass
_orig_stdout = sys.stdout
_orig_stderr = sys.stderr
sys.stdout = _Suppress()
sys.stderr = _Suppress()

try:
    import quantstats as qs
    import pandas as pd
finally:
    sys.stdout = _orig_stdout
    sys.stderr = _orig_stderr

# Suppress quantstats' inline display calls (iDisplay/iHTML) to avoid IPython JS errors
import quantstats.reports as _qs_reports
if hasattr(_qs_reports, 'iDisplay'):
    _qs_reports.iDisplay = lambda *args, **kwargs: None
if hasattr(_qs_reports, 'iHTML'):
    _qs_reports.iHTML = lambda x: x

# Also suppress the top-level IPython.display.display call
try:
    import IPython.display as _ip_disp
    _ip_disp.display = lambda *args, **kwargs: None
except Exception:
    pass

# --- Quantstats backtest report ---
best_pf = run_backtest(df, "rsi")
returns_series = best_pf.returns()
returns_series.name = "RSI Reversal"

# Use SPY as a proxy benchmark
try:
    import yfinance as yf
    benchmark = yf.download("SPY", start=returns_series.index[0], end=returns_series.index[-1])["Close"]
    benchmark.index.name = returns_series.index.name
    benchmark = benchmark.reindex(returns_series.index).ffill()
    benchmark_returns = benchmark.pct_change().reindex(returns_series.index).fillna(0)
except Exception:
    benchmark_returns = None

# Display summary metrics
print("=== Backtest Performance Summary ===")
stats = best_pf.stats()
for k, v in stats.items():
    print(f"{k}: {v}")

# Generate quantstats HTML report (file only — inline display suppressed)
print("\n=== Quantstats Report ===")
try:
    qs.reports.html(returns_series, benchmark=benchmark_returns, output="backtest_report.html", title="Backtest Report", download_filename=False)
    print("HTML report saved to: backtest_report.html")
except Exception as e:
    print(f"HTML report generation failed: {e}")
    print("Falling back to inline metrics...")
    qs.reports.metrics(returns_series, benchmark=benchmark_returns)

# Save comparison table to CSV as well
results_path = Path("backtest_results.csv")
table.to_csv(results_path)
print(f"\nResults saved to: {results_path}")

# Restore original IPython.display.display if it was patched
try:
    import IPython.display as _ip_disp
    _ip_disp.display.__wrapped__ = None  # no-op, safe to leave as lambda
except Exception:
    pass


[*********************100%***********************]  1 of 1 completed


=== Backtest Performance Summary ===
Start: 2024-10-15 00:00:00
End: 2026-10-15 00:00:00
Period: 507 days 00:00:00
Start Value: 100000.0
End Value: 112297.86631335129
Total Return [%]: 12.297866313351289
Benchmark Return [%]: 17.39634398383114
Max Gross Exposure [%]: 100.0
Total Fees Paid: 327.959402983279
Max Drawdown [%]: 18.163798788842016
Max Drawdown Duration: 132 days 00:00:00
Total Trades: 2
Total Closed Trades: 1
Total Open Trades: 1
Open Trade PnL: -1731.6711986029113
Win Rate [%]: 100.0
Best Trade [%]: 14.043567049466155
Worst Trade [%]: 14.043567049466155
Avg Winning Trade [%]: 14.043567049466155
Avg Losing Trade [%]: nan
Avg Winning Trade Duration: 72 days 00:00:00
Avg Losing Trade Duration: NaT
Profit Factor: inf
Expectancy: 14029.537511954202
Sharpe Ratio: 0.5680578568012306
Calmar Ratio: 0.47944274269220083
Omega Ratio: 1.1327186731340333
Sortino Ratio: 0.7885405604977238

=== Quantstats Report ===


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

HTML report saved to: backtest_report.html

Results saved to: backtest_results.csv
